> 请点击获取[课程 PPT 内容](https://www.canva.cn/design/DAGzTMVNGkU/aQmGVC-qPbugBYLZwdVchw/view?utm_content=DAGzTMVNGkU&utm_campaign=designshare&utm_medium=link2&utm_source=uniquelinks&utlId=h7bb343e643)。


# 1. 环境配置

## 1.1 python 环境准备

In [ ]:
! pip install gradio==6.2.0 openai==2.11.0 dashscope==1.25.4 langchain-classic==1.0.0 langchain==1.2.0 langchain-community==0.4.1 langchain-openai==1.1.6 agentevals==0.0.9 openevals==0.1.3

## 1.2 大模型密钥准备

请根据第一章内容获取相关平台的 API KEY，如若未在系统变量中填入，请将 API_KEY 信息写入以下代码（若已设置请忽略）：

In [ ]:
import os

# os.environ["OPENAI_API_KEY"] = "sk-xxxxxxxx"
# os.environ["DASHSCOPE_API_KEY"] = "sk-yyyyyyyy"

## 1.3 LangSmith 环境配置
我们需要先前往 LangSmith 的官网并进行注册登录。

登录后我们就进入了下面这个初始界面，此时我们需要找到左下角的 Setting ，然后在里面先获取新建一个 API Key。

创建完成后，我们就可以将其配置到环境变量中。除了 API_Key 以外，通常 LangSmith 的项目还需要设置是否跟踪、上传地址以及项目名称信息（这个需要自定义设置）。

In [ ]:
import os
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_API_KEY"] = "your langsmith api key"
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGSMITH_PROJECT"] = "ai-studio-evaluation"

# 2. 大模型测试

## 2.1 简介
对于智能体的测试，我们不能够仅仅看其是否完成正确答案，而是要更深入这个系统当中去看它是怎么得到这个答案的。这也是 LangChain Test 模块核心的设计思想，并非简单的单元测试（Unit Test），而是要更关注于集成测试（Integration Test）。

## 2.2 GenericFakeChatModel

GenericFakeChatModel 工具能够假装大模型的调用并返回对应格式的内容。

我们可以举一个最简单的例子，就在里面放入两个字符串，然后通过两次 model.invoke().content （返回 AIMessage 内容并获取内部 content）进行调用：

In [ ]:
from langchain_core.language_models.fake_chat_models import GenericFakeChatModel

model = GenericFakeChatModel(
    messages=iter([
        "hello",
        "world"])
)

print(model.invoke("anything").content)
print(model.invoke("anything").content)

此时无论传入的内容是什么，都只会输出 hello 和 world 两部分内容。所以这种方式很好的一点在于其是完全确定的，不需要依赖提示词、上下文以及其他参数。

## 2.3 行为轨迹评估器

在智能体的测试中，最重要的是关注其 Agent Trajectory（行为轨迹）。也就是说大模型最终回答对不对远远不够，我们必须测试“它是怎么做出这个回答的”？即便大模型给的回复最终是一样的，但是实现过程天差地别。

这些测试都是需要真实的模型来进行测试，Fake Model 是“演不出来的”。而在 LangChain 中专门推出了 agentevals 库来完成这方面的工作。

Agent Trajectory （行为轨迹）主要有两种评估的模式：
- 第一种是规则式（Trajectory Match Evaluator），其主要评估方式是：
    - 我提前知道“正确行为长什么样”
    - 拿实际轨迹逐步对比

- 另外一种是评审式（LLM-as-Judge Evaluator），其主要的评估方式是：
    - 不写死规则
    - 用另一个 LLM 来“评判是否合理”

### 2.3.1 Trajectory Match Evaluator
在 LangChain 中提供了四种 Trajectory Match 模式：

| 模式        | 行为约束                     | 适合场景                     |
|-------------|------------------------------|------------------------------|
| **strict**      | 顺序、工具调用完全一致           | 合规 / 审批 / 强流程            |
| **unordered**  | 工具集合一致，顺序不重要         | 信息收集                     |
| **subset**     | 不允许多余工具                 | 权限 / 安全                   |
| **superset**   | 至少做这些事                 | 最低行为保障                 |


#### 2.3.1.1 Strict Match
在这个模式下，每一步的工作调用以及顺序都需要和我们设定的目标是一致的，因此非常适合一些要求非常严格的场景下。

首先我们需要导入 create_trajectory_match_evaluator 并将模式设置为 strict：

In [ ]:
from agentevals.trajectory.match import create_trajectory_match_evaluator

evaluator = create_trajectory_match_evaluator(trajectory_match_mode="strict")  

然后我们需要准备前面提到的三部分内容：
- inputs：用户输入
- outputs：Agent 实际跑出来的 messages
- reference_outputs：你“认为正确的行为轨迹”

比如这里我们定义了一个函数，将输入设置为 "What's the weather in San Francisco?"。输出设置为调用输入后的结果 result：

In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain.messages import HumanMessage, AIMessage, ToolMessage
from agentevals.trajectory.match import create_trajectory_match_evaluator
from langchain_community.chat_models import ChatTongyi
import os

model = ChatTongyi(api_key=os.environ.get("DASHSCOPE_API_KEY"), model="qwen-max")

@tool
def get_weather(city: str):
    """Get weather information for a city."""
    return f"It's 75 degrees and sunny in {city}."

agent = create_agent(model, tools=[get_weather])

evaluator = create_trajectory_match_evaluator(  
    trajectory_match_mode="strict",  
)  

def test_weather_tool_called_strict():
    result = agent.invoke({
        "messages": [HumanMessage(content="What's the weather in San Francisco?")]
    })

    reference_trajectory = [
        HumanMessage(content="What's the weather in San Francisco?"),
        AIMessage(content="", tool_calls=[
            {"id": "call_1", "name": "get_weather", "args": {"city": "San Francisco"}}
        ]),
        ToolMessage(content="It's 75 degrees and sunny in San Francisco.", tool_call_id="call_1"),
        AIMessage(content="The weather in San Francisco is 75 degrees and sunny."),
    ]

    evaluation = evaluator(
        outputs=result["messages"],
        reference_outputs=reference_trajectory
    )
    print(evaluation)
    assert evaluation["score"] is True
    
test_weather_tool_called_strict()

在当前假工具的情况下，这种一般不会出错，但是严格模式下主要审查的点有几个：
- 消息数量是否一致，不一致直接判定为不通过
- 消息角色是否一一对应，输出结果与预期结果的角色不匹配则不通过
- 是否同时存在或同时不存在 tool_calls，仅一方存在则不通过
- tool_calls 数量是否一致，数量不一致则不通过
- tool_calls 中的工具名称是否一致，存在不匹配则不通过
- tool_calls 的参数值是否一致，参数不一致同样判定为不通过

基于以上规则，可以看到下面的示例中，虽然绝大部分的内容是一致的，但就是因为 tool_calls 里的内容不一样所以返回的 score 就是 False：

In [ ]:
import json
from agentevals.trajectory.match import create_trajectory_match_evaluator

outputs = [
    {"role": "user", "content": "What is the weather in San SF?"},
    {"role": "assistant", "content": "", "tool_calls": [
        {"function": {"name": "get_weather", "arguments": json.dumps({"city": "San Francisco"})}},
        {"function": {"name": "accuweather_forecast", "arguments": json.dumps({"city": "San Francisco"})}},
    ]},
    {"role": "tool", "content": "It's 80 degrees and sunny in SF."},
    {"role": "assistant", "content": "The weather in SF is 80 degrees and sunny."}]

reference_outputs = [
    {"role": "user", "content": "What is the weather in San Francisco?"},
    {"role": "assistant", "content": "", "tool_calls": [
        {"function": {"name": "get_weather", "arguments": json.dumps({"city": "San Francisco"})}},
    ]},
    {"role": "tool", "content": "It's 80 degrees and sunny in San Francisco."},
    {"role": "assistant", "content": "The weather in SF is 80˚ and sunny."},
]

evaluator = create_trajectory_match_evaluator(trajectory_match_mode="strict")

result = evaluator(outputs=outputs, reference_outputs=reference_outputs)

print(result)


但是假如两者的内容差别并不大，只不过是大小写的区别，比如 "san francisco" 和 "San Francisco" ，这个情况下假如我们还严格设定其为不一样的话，显然不太合理。
因此在评估器设置时，我们可以对 tool_args_match_mode 进行设置，比如：

In [ ]:
evaluator = create_trajectory_match_evaluator(
    trajectory_match_mode="strict",
    tool_args_match_mode="exact")  # 模型情况

tool_args_match_mode 有四种模式：
- exact（默认）：参数必须完全一致。该模式校验最为严格，只有当工具调用的参数结构和取值都高度稳定、且不希望 LLM 进行任何“自由补充或省略”时才适合使用。
- ignore：只要工具名称一致即可视为匹配。该模式完全忽略参数内容，仅关注工具调用是否发生以及调用顺序是否正确。
- subset：输出参数是参考参数的子集。
- superset：输出参数是参考参数的超集。

除了这四种模式以外，还有一种更高级的方法 tool_args_match_overrides ，其允许对某些工具单独定规则，其优先级比起前面四种模式都要高。
比如说最开始讲的 "san francisco" 和 "San Francisco" 例子，即便我们使用前面四种模式都是无法解决的，那这个时候我们可以自己来制定规则，比如：

In [ ]:
tool_args_match_overrides={"get_weather": lambda x, y: x["city"].lower() == y["city"].lower()}

此时就可以通过将 city 参数都转变为小写的，那这个时候再进行对比就会返回 True 了。

当然使用函数是比较复杂的用法，比较简单的用法比如指定某个工具的对应参数用某种模式的话，可以通过：

In [ ]:
tool_args_match_overrides={"get_weather": "ignore"}

在这个情况下，get_weather 这个工具就完全不会考虑参数里面具体内容的审查了。又比如我们只对比一部分的字段：

In [ ]:
tool_args_match_overrides={"get_weather": ["city"]}

那此时只要这里 city 的信息一样即可，其他参数不一样也没关系。

那 overrides 方法可以和正常方法进行一起写，系统会将 overrider 方法有的内容覆盖原有的方法，其他的都保持不变，比如：

In [ ]:
evaluator = create_trajectory_match_evaluator(
    trajectory_match_mode="strict",
    tool_args_match_mode="exact",  # Default value
    tool_args_match_overrides={
        "get_weather": lambda x, y: x["city"].lower() == y["city"].lower()
    }
)

此时审查的话，"san francisco" 和 "San Francisco" 即便不同也会 score 返回为 True 了。当然两段代码目前的问题不仅仅只是该信息不同，还有其他很多不一样的地方，所以返回的内容还是 False。

In [ ]:
result = evaluator(outputs=outputs, reference_outputs=reference_outputs)

print(result)

#### 2.3.1.2 Unordered Match
相比于 Strict 的形式，unordered 不再“逐步对齐消息”，而是“整体集合判断”。

比如在下面的示例中，在 outputs 里虽然是分开两次进行工具的调用，而在 reference_outputs 里则是一次性将两个工具进行调用：

In [ ]:
inputs = {}
outputs = [
    {"role": "user", "content": "What is the weather in SF and is there anything fun happening?"},
    {"role": "assistant", "content": "", "tool_calls": [
        {"function": {"name": "get_weather", "arguments": json.dumps({"city": "San Francisco"})}},
    ]},
    {"role": "tool", "content": "It's 80 degrees and sunny in SF."},
    {"role": "assistant", "content": "", "tool_calls": [
        {"function": {"name": "get_fun_activities", "arguments": json.dumps({"city": "San Francisco"})}},
    ]},
    {"role": "tool", "content": "Nothing fun is happening, you should stay indoors and read!"},
    {"role": "assistant", "content": "The weather in SF is 80 degrees and sunny, but there is nothing fun happening."},
]


假如这在 Strict 模式下就会判定为 False 了。但是在 Unordered 模式下由于调用的内容其实是一样的，所以就还是会返回 True：

In [ ]:
reference_outputs = [
    {"role": "user", "content": "What is the weather in SF and is there anything fun happening?"},
    {"role": "assistant", "content": "", "tool_calls": [
        {"function": {"name": "get_fun_activities", "arguments": json.dumps({"city": "San Francisco"})}},
        {"function": {"name": "get_weather", "arguments": json.dumps({"city": "San Francisco"})}},
    ]},
    {"role": "tool", "content": "Nothing fun is happening, you should stay indoors and read!"},
    {"role": "tool", "content": "It's 80 degrees and sunny in SF."},
    {"role": "assistant", "content": "In SF, it's 80˚ and sunny, but there is nothing fun happening."},
]

evaluator = create_trajectory_match_evaluator(trajectory_match_mode="unordered")
print(evaluator(outputs=outputs, reference_outputs=reference_outputs))

所以可以看出来，主要 unordered 主要审查的是工具是否正确调用了，而不去考虑顺序是否一致等等的内容，因此比较适合信息收集、搜索以及多来源查询等应用场景。在这些场景中，“有没有查天气”比“城市字符串是否完全一致”更重要。

#### 2.3.1.3 Subset Match

和 Unordered 模式类似，Subset 也是只比较 tool_call 多重集合。但是 Subset 模式的核心区别在于“是否越权”。

Subset 的 evaluator 的创建方法也很简单，就是修改名称即可：

In [ ]:
evaluator = create_trajectory_match_evaluator(
    trajectory_match_mode="subset", 
)

#### 2.3.1.4 Superset Match
Superset 与 Unordered / Subset 一样，仅关注工具调用本身，而不关心上下文细节。但在判定规则上，Superset 与 Subset 正好相反。Subset 允许少做事，但不允许多做事；而 Superset 则允许多做事，但不允许少做事。

Superset 的 evaluator 的创建方法也很简单，就是修改名称即可：

In [ ]:
evaluator = create_trajectory_match_evaluator(
    trajectory_match_mode="superset", 
)

### 2.3.2 LLM-as-Judge Evaluator

除了通过规则的方式去评价以外，其实我们还可以考虑通过大模型来进行评价整个流程是否正常合理。

通过规则评判的前提是你能“提前写出规则”，但现实里，经常是下面这种情况：
- Agent 的推理步骤会变
- 工具调用顺序会变
- 有时候多一步，有时候少一步
- 但整体行为仍然是“合理的”

这时规则式 matcher 会失败，但人类会说“这没问题”。因此为了解决这一类的问题，我们可以通过更专业或者说更强大的模型模拟“人类评审员”。

在 LangChain 中内置了一段系统提示词完成该部分任务：

In [ ]:
TRAJECTORY_ACCURACY_PROMPT = """You are an expert data labeler.
Your task is to grade the accuracy of an AI agent's internal trajectory.
<Rubric>
  An accurate trajectory:
  - Makes logical sense between steps
  - Shows clear progression
  - Is relatively efficient, though it does not need to be perfectly efficient
</Rubric>
First, try to understand the goal of the trajectory by looking at the input
(if the input is not present try to infer it from the content of the first message),
as well as the output of the final message. Once you understand the goal, grade the trajectory
as it relates to achieving that goal.
Grade the following trajectory:
<trajectory>
{outputs}
</trajectory>
"""

为了能够实现让大模型进行判断，我们首先需要先定义一个判断的大模型，比如这里我们还是使用 ChatTongyi 来进行完成，然后提示词就使用前面展示的提示词：

In [ ]:
from agentevals.trajectory.llm import create_trajectory_llm_as_judge, TRAJECTORY_ACCURACY_PROMPT

model = ChatTongyi(api_key=os.environ.get("DASHSCOPE_API_KEY"), model="qwen-turbo")
evaluator = create_trajectory_llm_as_judge(prompt=TRAJECTORY_ACCURACY_PROMPT, judge=model)

然后我们就可以来设置一个函数来传入 outputs 的内容并且直接让大模型评判 outputs 的内容是否合理（不需要传入 references 了！）：

In [ ]:
def test_trajectory_quality():
    result = agent.invoke({"messages": [HumanMessage(content="What's the weather in Seattle?")]})
    evaluation = evaluator(outputs=result["messages"])
    print(evaluation)
    assert evaluation["score"] is True
test_trajectory_quality()

当然这里演示的是没有 references 的情况下，假如我们要加上 references 答案的话一样也是 ok 的，只不过我们需要先更换一下模型提示词为 TRAJECTORY_ACCURACY_PROMPT_WITH_REFERENCE，也就是：

In [ ]:
TRAJECTORY_ACCURACY_PROMPT_WITH_REFERENCE = """You are an expert data labeler.
Your task is to grade the accuracy of an AI agent's internal trajectory.
<Rubric>
  An accurate trajectory:
  - Makes logical sense between steps
  - Shows clear progression
  - Is relatively efficient, though it does not need to be perfectly efficient
  - Is semantically equivalent to the provided reference trajectory
</Rubric>
Based on the following reference trajectory:
<reference_trajectory>
{reference_outputs}
</reference_trajectory>
Grade this actual trajectory:
<trajectory>
{outputs}
</trajectory>
"""

这里其实也就是把参考答案加进去了而已，所以模型和之前是一样的：

In [ ]:
from agentevals.trajectory.llm import TRAJECTORY_ACCURACY_PROMPT_WITH_REFERENCE

evaluator = create_trajectory_llm_as_judge(judge=model,
    prompt=TRAJECTORY_ACCURACY_PROMPT_WITH_REFERENCE,)

然后呢在评估的时候我们也需要更新一下，不再单纯传入的是 outputs 的内容了，而是要把参考的路径也给进去，比如：

```python
evaluation = judge_with_reference(outputs=result["messages"],
    reference_outputs=reference_trajectory)
```

在创建 create_trajectory_llm_as_judge 时（直接使用的是 openevals 仓库中的 create_llm_as_judge 方法），除了上面提到的 prompt 和 model 两个参数，其实还有其他的参数，包括：

```python
scorer = create_trajectory_llm_as_judge(
    prompt=prompt,              # 评审规则 +任务说明
    judge=judge,                # 谁来评？
    model=model,                # 用哪个模型来评？
    continuous=continuous,      # 打连续分还是二值？
    choices=choices,            # 分数档位
    use_reasoning=use_reasoning,# 要不要解释
    few_shot_examples=few_shot_examples,  # 示例教学
)
```